# サブワード正則化: 分割をサンプリングしてデータ拡張する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/06_subword/subword_regularization.ipynb)

NLP クックブック 06-02 ／ サブワードとトークナイザ

同じ文を毎回違う分割でモデルに見せて、頑健性を上げたい (subword regularization / BPE-dropout)。

In [1]:
!pip install -q sentencepiece

In [2]:
import sentencepiece as spm

import io, re, urllib.request, zipfile

# 青空文庫「吾輩は猫である」でモデルを学習 (前のレシピと同じ)
url = "https://www.aozora.gr.jp/cards/000148/files/789_ruby_5639.zip"
with urllib.request.urlopen(url) as r:
    z = zipfile.ZipFile(io.BytesIO(r.read()))
text = z.read(z.namelist()[0]).decode("cp932")
text = re.split(r"-{50,}", text)[2].split("底本：")[0]
text = re.sub(r"《.+?》|［＃.+?］|｜", "", text)
with open("corpus.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(s.strip() for s in re.split(r"(?<=。)|\r\n", text) if s and s.strip()))

spm.SentencePieceTrainer.train(input="corpus.txt", model_prefix="neko", vocab_size=4000, model_type="unigram", minloglevel=2)
sp = spm.SentencePieceProcessor(model_file="neko.model")

In [3]:
text = "吾輩は猫である。名前はまだ無い。"
print("最尤:", sp.encode(text, out_type=str))
for _ in range(5):
    print("標本:", sp.encode(text, out_type=str, enable_sampling=True, alpha=0.2, nbest_size=-1))

最尤: ['▁吾輩は', '猫', 'である', '。', '名', '前', 'は', 'まだ', '無', 'い', '。']
標本: ['▁', '吾輩は', '猫', 'である', '。', '名', '前', 'は', 'まだ', '無', 'い', '。']
標本: ['▁', '吾', '輩', 'は', '猫', 'である', '。', '名', '前', 'は', 'まだ', '無', 'い', '。']
標本: ['▁吾輩は', '猫', 'である', '。', '名', '前', 'は', 'まだ', '無', 'い', '。']
標本: ['▁吾輩は', '猫', 'である', '。', '名', '前', 'は', 'まだ', '無', 'い', '。']
標本: ['▁吾輩は', '猫', 'で', 'ある', '。', '名', '前', 'は', 'まだ', '無', 'い', '。']


In [4]:
# 確率の高い順に N 個の分割候補
for pieces in sp.nbest_encode(text, nbest_size=3, out_type=str):
    print(pieces)

['▁吾輩は', '猫', 'である', '。', '名', '前', 'は', 'まだ', '無', 'い', '。']
['▁吾輩は', '猫', 'である', '。', '名', '前', 'は', 'ま', 'だ', '無', 'い', '。']
['▁', '吾輩は', '猫', 'である', '。', '名', '前', 'は', 'まだ', '無', 'い', '。']


- `alpha` が小さいほど多様な (確率の低い) 分割が出る。`nbest_size=-1` は全候補からサンプリング。
- 学習時だけサンプリングし、評価時は最尤分割を使う。